# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)

d

# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [0]:
# Write your solution here
from pyspark.sql.functions import *
# Extract
bookings_df = spark.table("bookings")

# Transform
result_df = (bookings_df.filter((col("starttime") >= "2012-09-01") &(col("starttime") < "2012-10-01")).\
    groupBy("facid").\
    agg(sum("slots").alias("Total Slots")).\
        orderBy("Total Slots")
)

# Load
output_path = "/FileStore/tables/etl_job_one_parquet"
result_df.write.mode("overwrite").parquet(output_path)

## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
# Write your solution here
# Extract
members_df = spark.table("members")
facilities_df = spark.table("facilities")

b = bookings_df.alias("b")
m = members_df.alias("m")
f = facilities_df.alias("f")

# Transform
result_df = m.join(b, col("m.memid") == col("b.memid"), "inner").\
    join(f, col("b.facid") == col("f.facid"), "inner").\
        filter(col("f.name").isin("Tennis Court 1", "Tennis Court 2")).\
            select(concat_ws(" ", col("m.firstname"), col("m.surname")).alias("member"),col("f.name").alias("facility")).\
                distinct().\
                    orderBy("member", "facility")

result_df.show()

# Load
result_df.write.mode("overwrite").partitionBy("facility").saveAsTable("threejoin_delta")



+--------------+--------------+
|        member|      facility|
+--------------+--------------+
|    Anne Baker|Tennis Court 1|
|    Anne Baker|Tennis Court 2|
|  Burton Tracy|Tennis Court 1|
|  Burton Tracy|Tennis Court 2|
|  Charles Owen|Tennis Court 1|
|  Charles Owen|Tennis Court 2|
|  Darren Smith|Tennis Court 2|
| David Farrell|Tennis Court 1|
| David Farrell|Tennis Court 2|
|   David Jones|Tennis Court 1|
|   David Jones|Tennis Court 2|
|  David Pinker|Tennis Court 1|
| Douglas Jones|Tennis Court 1|
| Erica Crumpet|Tennis Court 1|
|Florence Bader|Tennis Court 1|
|Florence Bader|Tennis Court 2|
|   GUEST GUEST|Tennis Court 1|
|   GUEST GUEST|Tennis Court 2|
|Gerald Butters|Tennis Court 1|
|Gerald Butters|Tennis Court 2|
+--------------+--------------+
only showing top 20 rows


## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:
# Write your solution here
import requests
from pyspark.sql.functions import col, to_date, weekofyear, year, max as spark_max
from functools import reduce

url = "https://alpha-vantage.p.rapidapi.com/query"

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "YOUR_RAPIDAPI_KEY"
}

companies = {
    "Google": "GOOGL",
    "Apple": "AAPL",
    "Microsoft": "MSFT",
    "Tesla": "TSLA"
}

all_dfs = []

# Extract
for company, symbol in companies.items():
    querystring = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "datatype": "json",
        "outputsize": "compact"
    }

    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()
    time_series = data["Time Series (Daily)"]

    rows = []

    rows.append((company,symbol,date,float(values["4. close"])))

    df = spark.createDataFrame(rows,["company", "symbol", "date", "closing_price"])

    all_dfs.append(df)

stock_df = reduce(lambda df1, df2: df1.unionByName(df2), all_dfs)

# Transform
stock_df = stock_df.withColumn("date", to_date(col("date"))).\
    withColumn("year", year(col("date"))).\
        withColumn("week", weekofyear(col("date")))

result_df = stock_df.groupBy("company", "symbol", "year", "week").\
    agg(spark_max("closing_price").alias("max_closing_price")).\
        orderBy("company", "year", "week")


result_df.show()

# Load
result_df.write.mode("overwrite").partitionBy("company").saveAsTale("max_closing_price_weekly")



---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
File <command-5052699759982392>, line 33
     31 response = requests.get(url, headers=headers, params=querystring)
     32 data = response.json()
---> 33 time_series = data["Time Series (Daily)"]
     35 rows = []
     37 rows.append((company,symbol,date,float(values["4. close"])))

KeyError: 'Time Series (Daily)'

## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:
# Write your solution here
# Extract from PostgreSQL using JDBC

rna_df = (
    spark.read
    .format("jdbc")
    .option("url","jdbc:postgresql://hh-pgsql-public.ebi.ac.uk:5432/pfmegrnargs")
    .option("dbtable","(SELECT * FROM rna LIMIT 100) AS rna_100")
    .option("user", "reader")
    .option("password", "NWDMCE5xdipIjRrp")
    .option("driver", "org.postgresql.Driver")
    .load()
)

rna_df.show()

# Load 
rna_df.write.mode("overwrite").saveAsTable("rna_100_records")


+--------+-------------+--------------------+---------+----------------+----+--------------------+--------+--------------------+
|      id|          upi|           timestamp|userstamp|           crc64| len|           seq_short|seq_long|                 md5|
+--------+-------------+--------------------+---------+----------------+----+--------------------+--------+--------------------+
|29027282|URS0001BAEBD2|2020-08-02 23:55:...|   rnacen|3EE097D39F1A5BCF|1777|TCTGGTTGATCCTGCCA...|    NULL|f673c8dcd23252a48...|
|29027283|URS0001BAEBD3|2020-08-02 23:55:...|   rnacen|7C80F6262CA92AF0|1260|AGCTAGTTGGTAGGGTA...|    NULL|f673cfbb4f4d79806...|
|29027284|URS0001BAEBD4|2020-08-02 23:55:...|   rnacen|2E7AF64F01FD06D5|1407|AGAGTTTGATCCCGGCT...|    NULL|f673d370236dc3498...|
|29027285|URS0001BAEBD5|2020-08-02 23:55:...|   rnacen|594BC3631CD742DE|1318|GGATAAACCCGGGAAAC...|    NULL|f673d768b475e05cb...|
|29027286|URS0001BAEBD6|2020-08-02 23:55:...|   rnacen|B937E00FE157EDA7|1499|GAGTTTGATCATGGCTC...